# VCPI Drug-seq Hackathon — Colab Setup

This notebook loads pre-processed training data from GCS (no re-download from vcpi-client needed).

**Before running:**
1. In the Colab sidebar, click the 🔑 key icon → Add a secret named `TVC_TOKEN` → paste your token → enable notebook access
2. Make sure Runtime → Change runtime type → T4 GPU is selected
3. Run cells top to bottom

In [ ]:
# ── Cell 1: Install packages ──────────────────────────────────────────────────
# Run once per Colab session (~2 min)
!pip install \
    git+https://github.com/virtualcell-vcpi/vcpi-client.git \
    git+https://github.com/virtualcell-vcpi/vcpi-prediction-contest-2026.git \
    google-cloud-storage \
    polars pyarrow \
    rdkit seaborn scikit-learn scipy \
    --quiet
print('Packages installed.')

In [ ]:
# ── Cell 2: Set TVC_TOKEN from Colab Secrets ──────────────────────────────────
import os
from google.colab import userdata

os.environ['TVC_TOKEN'] = userdata.get('TVC_TOKEN')
assert os.environ.get('TVC_TOKEN'), 'TVC_TOKEN not set — check Colab Secrets'
print('TVC_TOKEN set.')

In [ ]:
# ── Cell 3: Authenticate to GCS ───────────────────────────────────────────────
from google.colab import auth
auth.authenticate_user()  # opens OAuth popup — use maya.levy@gensaic.com
print('GCS auth complete.')

In [ ]:
# ── Cell 4: GCS loader helper ─────────────────────────────────────────────────
import io
import polars as pl
from google.cloud import storage

BUCKET = 'vcpi-drugseq-2026'
_client = None

def gcs():
    global _client
    if _client is None:
        _client = storage.Client()
    return _client

def load_parquet(blob_path: str) -> pl.DataFrame:
    buf = io.BytesIO()
    gcs().bucket(BUCKET).blob(blob_path).download_to_file(buf)
    buf.seek(0)
    return pl.read_parquet(buf)

print('Loader ready. Available datasets: tvc-bhr-009, tvc-kdl-010, tvc-qnu-012')

In [ ]:
# ── Cell 5: Load data from GCS ────────────────────────────────────────────────
# Load one dataset at a time to stay within Colab RAM (~12 GB)
# Adjust which datasets to load as needed

print('Loading tvc-bhr-009...')
counts_009 = load_parquet('data/tvc-bhr-009/counts.parquet')
meta_009   = load_parquet('data/tvc-bhr-009/metadata.parquet')
chem_009   = load_parquet('data/tvc-bhr-009/chemistry.parquet')
print(f'  counts: {counts_009.shape}, meta: {meta_009.shape}, chem: {chem_009.shape}')

# Uncomment to load additional datasets:
# print('Loading tvc-kdl-010...')
# counts_010 = load_parquet('data/tvc-kdl-010/counts.parquet')
# meta_010   = load_parquet('data/tvc-kdl-010/metadata.parquet')
# chem_010   = load_parquet('data/tvc-kdl-010/chemistry.parquet')

# print('Loading tvc-qnu-012...')
# counts_012 = load_parquet('data/tvc-qnu-012/counts.parquet')
# meta_012   = load_parquet('data/tvc-qnu-012/metadata.parquet')
# chem_012   = load_parquet('data/tvc-qnu-012/chemistry.parquet')

In [ ]:
# ── Cell 6: Check GPU ─────────────────────────────────────────────────────────
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else 'No GPU detected — check Runtime type')

In [ ]:
# ── Cell 7: Contest utilities ─────────────────────────────────────────────────
from vcpi_prediction_contest import (
    counts_to_expression,
    load_gene_filter,
    load_test_compounds,
    score_compounds,
    predict_mu_all_train,
)

gene_filter    = load_gene_filter()     # 12,995 scored genes
test_compounds = load_test_compounds()  # 1,064 test compounds with SMILES

print(f'Gene filter: {len(gene_filter)} genes')
print(f'Test compounds: {len(test_compounds)}')
print(f'\nTest compound columns: {test_compounds.columns.tolist()}')
print(test_compounds.head(3).to_string())

In [ ]:
# ── Cell 8: Quick baseline (per-gene mean) ────────────────────────────────────
# Convert counts → expression for -009
expr_009 = counts_to_expression(counts_009.to_pandas(), meta_009.to_pandas())
print(f'Expression shape (long): {expr_009.shape}')
print(expr_009.head(3))

# Baseline: per-gene mean prediction for all test compounds
baseline_pred = predict_mu_all_train(
    truth_train=expr_009,
    test_compounds=test_compounds['compound'].tolist(),
    gene_filter=gene_filter,
)
print(f'\nBaseline prediction shape: {baseline_pred.shape}')
print('Columns:', baseline_pred.columns.tolist())

## Next steps
- Load all three datasets, combine `expr_009 / expr_010 / expr_012` into one training frame
- Extract Morgan fingerprints from `chem_009['smiles']` via rdkit
- Train MLP: fingerprint → per-gene expression vector
- Predict for test compounds, format submission as parquet with `[compound, gene_id, predicted_expression]`
- Score locally: `score_compounds(truth, prediction, weights=canonical_weights)`